# E1 complete-pose receiving
Owner: uibcdf/dockingmt#78. Execute only with registered inputs, environment and resource monitor.
Builds new local H per pose; no Vina/scoring or historical prepared-state restoration.


In [ ]:
import base64
import gzip
import hashlib
import importlib.metadata as metadata
import json
import os
import platform
import sys
from pathlib import Path

root = Path(os.environ['DOCKINGMT_E1_ROOT'])
output = Path(os.environ['DOCKINGMT_E1_COMPLETE_OUTPUT'])
output.mkdir(parents=True, exist_ok=False)
inputs = {
    'source': root
    / 'devguide/validation/data/e1_declared_authority_2026-11-06/normal/declared_authority9.h5msm.gz',
    'variants': root
    / 'devguide/validation/data/e1_declared_authority_2026-11-06/normal/observations.json',
    'legacy': root
    / 'devguide/validation/data/human_3ptb/native_result_2026-10-20.json.gz',
    'helper': root / 'devguide/validation/e1_prepared_authority.ipynb',
}
registered_inputs = json.loads(os.environ['DOCKINGMT_E1_INPUT_SHA256'])
input_bytes = {}
for name, path in inputs.items():
    data = path.read_bytes()
    assert hashlib.sha256(data).hexdigest() == registered_inputs[name]
    input_bytes[name] = data
assert (
    registered_inputs['helper']
    == '31993b57ad1377ba7605dd17d5e769686289ea7c183be8d57a8350294c1d5972'
)
assert (
    registered_inputs['legacy']
    == '4b1f51cdeb135fd6785a181f6bb09a0d0f631ec3d9de7c169c88703748924249'
)
source_native = gzip.decompress(input_bytes['source'])
assert (
    hashlib.sha256(source_native).hexdigest()
    == 'fd01a684d624e8abaa19ee6d16d62ab0b84927c3abd74d5bbd5dcc3881c712b7'
)
source_path = output / 'declared_authority9.h5msm'
source_path.write_bytes(source_native)
variants = json.loads(input_bytes['variants'])['pose_variants']
legacy = json.loads(gzip.decompress(input_bytes['legacy']))
assert len(variants) == len(legacy['poses']) == 5
for index, row in enumerate(variants):
    assert (
        row['original_position'] == index + 1
        and row['pose_id'] == legacy['poses'][index]['pose_id']
    )
    assert (
        row['original_scores'] == legacy['poses'][index]['scores']
        and row['coordinates']['unit'] == 'nm'
    )
receptor = base64.b64decode(
    legacy['provenance']['backend_artifacts']['receptor']['content_base64'],
    validate=True,
)
assert (
    hashlib.sha256(receptor).hexdigest()
    == 'edeff77e6222c5c5ab1aa6be2354efc323c0ffe8a7884366069ee0addbf7b970'
)
(output / 'provided_receptor.pdbqt').write_bytes(receptor)
helper_cell = next(
    c
    for c in json.loads(input_bytes['helper'])['cells']
    if c['id'] == 'e1-prepared-helpers'
)
helper_source = ''.join(helper_cell['source'])
assert (
    hashlib.sha256(helper_source.encode()).hexdigest()
    == os.environ['DOCKINGMT_E1_HELPER_SHA256']
)

In [ ]:
# Exact previously received public-API inspection helpers; no molecular algorithm.
from functools import wraps
from types import SimpleNamespace

import molsysmt as msm
import numpy as np
import pyunitwizard as puw

import dockingmt as dmt

helper_namespace = dict(globals(), wraps=wraps, SimpleNamespace=SimpleNamespace)
exec(
    compile(helper_source, 'e1_prepared_authority.received_helpers', 'exec'),
    helper_namespace,
)
(
    api,
    save,
    xyz,
    charges,
    model_snapshot,
    mechanics_inspection,
    chemical_only_file,
    witness,
    fingerprint,
) = [
    helper_namespace[name]
    for name in [
        'api',
        'save',
        'xyz',
        'charges',
        'model_snapshot',
        'mechanics_inspection',
        'chemical_only_file',
        'witness',
        'fingerprint',
    ]
]
budgets = dict(
    convert=180,
    get=850,
    copy=20,
    set=40,
    permute_atoms=0,
    add_missing_hydrogens=10,
    assign_partial_charges=10,
    assign_autodock_atom_types=10,
    prepare_ligand=10,
    get_hydrogen_inventory=24,
    get_atom_correspondences=30,
    compose_atom_maps=10,
    read_poses=10,
    get_torsion_tree=20,
)
counts = dict.fromkeys(budgets, 0)
helper_namespace.update(budgets=budgets, counts=counts)
profiles = [
    (
        'normal',
        [
            'nm',
            'ps',
            'kelvin',
            'mole',
            'dalton',
            'e',
            'kJ/mol',
            'kJ/(mol*nm)',
            'kJ/(mol*nm**2)',
            'radians',
        ],
    ),
    (
        'pm_fs',
        [
            'pm',
            'fs',
            'kelvin',
            'mole',
            'dalton',
            'coulomb',
            'kJ/mol',
            'kJ/(mol*pm)',
            'kJ/(mol*pm**2)',
            'radians',
        ],
    ),
]
caller_policy = puw.configure.report()
observations = []
physical_values = []
center = np.asarray(legacy['provenance']['backend_box']['center'], dtype=float)
original_size = np.asarray(legacy['provenance']['backend_box']['size'], dtype=float)
size = np.asarray([24.0, 24.0, 24.0])
assert legacy['provenance']['backend_box']['unit'] == 'angstrom'
np.testing.assert_array_equal(center, [-1.759111, 14.461, 16.915778])
np.testing.assert_array_equal(original_size, [20.0, 20.0, 20.0])

In [ ]:
for profile, units in profiles:
    directory = output / profile
    directory.mkdir()
    with puw.context(standard_units=units):
        profile_policy = puw.configure.report()
        base = api.convert(source_path, to_form='molsysmt.MolSys')
        base_before = model_snapshot(base)
        inventory9 = api.get_hydrogen_inventory(
            base, structure_indices=[0], chemical_state='reference'
        )
        assert (
            inventory9['status'] == 'available'
            and np.sum(inventory9['missing_hydrogen_counts']) == 9
        )
        rows = []
        reference18 = reference13 = None
        for row in variants:
            index = row['original_position']
            pose_dir = directory / f'pose_{index}'
            pose_dir.mkdir()
            heavy = api.copy(base)
            coordinates = puw.quantity(
                row['coordinates']['value'], row['coordinates']['unit']
            )
            assert coordinates.shape == (1, 9, 3)
            api.set(
                heavy, element='atom', structure_indices=[0], coordinates=coordinates
            )
            np.testing.assert_allclose(
                xyz(heavy), row['coordinates']['value'], rtol=0, atol=1e-12
            )
            heavy_before = model_snapshot(heavy)
            assert heavy_before['chemical'] == base_before['chemical']
            expanded = api.add_missing_hydrogens(
                heavy,
                mode='fixed_chemical_state',
                pH=None,
                engine='RDKit',
                attribute_policy='strict',
                chemical_state='reference',
                structure_indices=[0],
                return_report=True,
            )
            full18, h_report = expanded['molecular_system'], expanded['report']
            save(pose_dir / 'hydrogen_addition.json', h_report)
            assert (
                h_report['n_added_hydrogens'] == 9
                and not h_report['dropped_attributes']
                and h_report['original_coordinates_preserved']
            )
            np.testing.assert_array_equal(
                h_report['atom_correspondence'],
                np.column_stack((np.arange(9), np.arange(9))),
            )
            pairs = np.asarray(h_report['parent_hydrogen_pairs'])
            assert pairs.shape == (9, 2) and np.all(
                (pairs[:, 0] >= 0) & (pairs[:, 0] < 9)
            )
            np.testing.assert_array_equal(np.sort(pairs[:, 1]), np.arange(9, 18))
            for attribute, expected in [
                ('atom_id', heavy_before['atom_ids']),
                ('name', heavy_before['atom_names']),
                ('atom_type', heavy_before['elements']),
            ]:
                np.testing.assert_array_equal(
                    api.get(full18, element='atom', **{attribute: True})[:9], expected
                )
            np.testing.assert_allclose(
                xyz(full18)[:, :9], row['coordinates']['value'], rtol=0, atol=1e-12
            )
            inventory18 = api.get_hydrogen_inventory(
                full18, structure_indices=[0], chemical_state='reference'
            )
            assert (
                inventory18['status'] == 'available'
                and len(inventory18['explicit_hydrogen_atom_indices']) == 9
                and np.sum(inventory18['missing_hydrogen_counts']) == 0
            )
            save(pose_dir / 'inventory18.json', inventory18)
            full_before = model_snapshot(full18)
            charged = api.assign_partial_charges(
                full18,
                method='gasteiger_marsili',
                chemical_state='reference',
                structure_indices=[0],
                return_report=True,
            )
            typed = api.assign_autodock_atom_types(
                charged['molecular_system'],
                typing_scheme='autodock4',
                method='chemical_environment',
                chemical_state='reference',
                structure_indices=[0],
                return_report=True,
            )
            typed18 = typed['molecular_system']
            save(pose_dir / 'charge_assignment.json', charged['report'])
            save(pose_dir / 'typing_assignment.json', typed['report'])
            typed_before = model_snapshot(typed18)
            assert typed_before['chemical'] == full_before['chemical']
            q18 = charges(typed18)
            types18 = list(api.get(typed18, element='atom', atom_ff_type=True))
            assert (
                np.isfinite(q18).all()
                and abs(np.sum(q18) - 1) <= 1e-6
                and types18.count('H') == 5
                and types18.count('HD') == 4
            )
            if reference18 is None:
                reference18 = typed18
            save(pose_dir / 'full18_witness.json', witness(typed18, reference18))
            np.testing.assert_allclose(q18, charges(reference18), rtol=0, atol=1e-12)
            np.testing.assert_array_equal(
                types18, api.get(reference18, element='atom', atom_ff_type=True)
            )
            chemical_only_file(typed18, pose_dir / 'full18_chemical_only.h5msm')
            save(
                pose_dir / 'full18_mechanics_inspection.json',
                mechanics_inspection(typed18),
            )
            prepared = api.prepare_ligand(
                typed18,
                selection='all',
                active_torsion_bonds=[(0, 6)],
                state_id='e1-new-declared-benzamidinium-20261106',
            )
            assert (
                prepared.n_atoms == 13
                and prepared.torsion_dof == 1
                and dmt.assess_preparation(prepared)['assessment'] == 'unassessed'
            )
            audit = dmt.audit_preparation_charges(prepared)
            assert audit['assessment'] == 'consistent'
            native13 = prepared.to_molecular_system()
            native_before = model_snapshot(native13)
            projection = prepared.metadata['molecular_projection']
            assert (
                projection['n_source_atoms'] == 18
                and projection['n_output_atoms'] == 13
                and len(projection['merged_source_atom_pairs']) == 5
            )
            assert (
                abs(
                    projection['output_total_charge']
                    - projection['selected_total_charge']
                )
                <= projection['charge_conservation_tolerance']
            )
            retained = np.asarray(projection['output_to_source_atom_indices'])
            np.testing.assert_allclose(
                xyz(native13), xyz(typed18)[:, retained], rtol=0, atol=1e-12
            )
            inventory13 = api.get_hydrogen_inventory(
                native13, structure_indices=[0], chemical_state='reference'
            )
            assert (
                inventory13['status'] == 'available'
                and len(inventory13['explicit_hydrogen_atom_indices']) == 4
                and np.sum(inventory13['missing_hydrogen_counts']) == 5
            )
            save(pose_dir / 'inventory13.json', inventory13)
            if reference13 is None:
                reference13 = native13
            save(pose_dir / 'prepared13_witness.json', witness(native13, reference13))
            np.testing.assert_allclose(
                charges(native13), charges(reference13), rtol=0, atol=1e-12
            )
            np.testing.assert_array_equal(
                prepared.atom_types,
                api.get(reference13, element='atom', atom_ff_type=True),
            )
            np.testing.assert_array_equal(charges(native13), prepared.charges)
            save(pose_dir / 'prepared.json', prepared.to_dict())
            save(pose_dir / 'charge_audit.json', audit)
            save(
                pose_dir / 'prepared_values.json',
                dict(
                    coordinates=prepared.coordinates,
                    partial_charge=puw.quantity(prepared.charges, 'elementary_charge'),
                    atom_ff_type=prepared.atom_types,
                ),
            )
            save(
                pose_dir / 'prepared_mechanics_inspection.json',
                mechanics_inspection(native13),
            )
            chemical_only_file(native13, pose_dir / 'prepared13_chemical_only.h5msm')
            order = list(range(6, 13)) + list(range(6))
            assert prepared.pdbqt_atom_indices == order
            tree = dict(
                schema_version='molsysmt.pdbqt-torsion-tree@1',
                atom_ids=api.get(native13, element='atom', atom_id=True),
                fragment_atom_indices=order,
                fragment_offsets=[0, 7, 13],
                branch_atom_pairs=np.asarray([[6, 0]]),
                branch_fragment_pairs=np.asarray([[0, 1]]),
                root_fragment_index=0,
                torsdof=1,
            )
            presentation = dict(
                atom_serials='consecutive',
                atom_names=prepared.atom_names,
                group_names=prepared.group_names,
                group_ids=[str(v) for v in prepared.group_ids],
                chain_ids='A',
                occupancy=1.0,
                b_factor=puw.quantity(0.0, 'angstrom**2'),
            )
            text, mapping = api.convert(
                native13,
                to_form='string:pdbqt_text',
                typing_scheme='autodock4',
                torsion_tree=tree,
                output_presentation=presentation,
                return_atom_map=True,
            )
            payload = text.removeprefix('pdbqt_text:')
            assert (
                mapping['payload_sha256']
                == hashlib.sha256(payload.encode()).hexdigest()
            )
            np.testing.assert_array_equal(
                mapping['output_to_source_atom_indices'], order
            )
            (pose_dir / 'native.pdbqt').write_text(payload)
            save(pose_dir / 'export_map.json', mapping)
            composed = api.compose_atom_maps([h_report, projection, mapping])
            assert (
                composed['n_source_atoms'] == 9
                and composed['n_output_atoms'] == 13
                and np.sum(np.asarray(composed['output_to_source_atom_indices']) == -1)
                == 4
            )
            save(pose_dir / 'source9_to_written_map.json', composed)
            dmt_payload = prepared.to_pdbqt()
            (pose_dir / 'dockingmt.pdbqt').write_text(dmt_payload)
            parsed_tree = api.get_torsion_tree(text)
            assert fingerprint(parsed_tree) == fingerprint(
                api.get_torsion_tree('pdbqt_text:' + dmt_payload)
            )
            save(pose_dir / 'written_tree.json', parsed_tree)
            decoded = api.convert(
                text, to_form='molsysmt.MolSys', discard_torsion_tree=True
            )
            decoded_dmt = api.convert(
                'pdbqt_text:' + dmt_payload,
                to_form='molsysmt.MolSys',
                discard_torsion_tree=True,
            )
            for attribute in ['atom_id', 'name', 'atom_type', 'atom_ff_type']:
                np.testing.assert_array_equal(
                    api.get(decoded, element='atom', **{attribute: True}),
                    api.get(decoded_dmt, element='atom', **{attribute: True}),
                )
            np.testing.assert_allclose(
                xyz(decoded), xyz(native13)[:, order], rtol=0, atol=0.0000500001
            )
            np.testing.assert_array_equal(xyz(decoded), xyz(decoded_dmt))
            np.testing.assert_array_equal(charges(decoded), charges(decoded_dmt))
            np.testing.assert_array_equal(
                charges(decoded), mapping['written_partial_charge']
            )
            np.testing.assert_allclose(
                charges(decoded),
                np.asarray(prepared.charges)[order],
                rtol=0,
                atol=0.000500001,
            )
            assert (
                abs(np.sum(charges(decoded)) - np.sum(prepared.charges))
                <= mapping['charge_rounding_error_bound'] + 1e-12
            )
            xyz_angstrom = puw.get_value(
                api.get(native13, element='atom', coordinates=True), to_unit='angstrom'
            )
            margin = float(
                np.min(
                    np.minimum(
                        xyz_angstrom - (center - size / 2),
                        (center + size / 2) - xyz_angstrom,
                    )
                )
            )
            original_margin = float(
                np.min(
                    np.minimum(
                        xyz_angstrom - (center - original_size / 2),
                        (center + original_size / 2) - xyz_angstrom,
                    )
                )
            )
            assert margin > 0.000500001
            reread = api.read_poses(text, native13, mapping, return_report=True)
            save(pose_dir / 'read_poses_report.json', reread['report'])
            save(
                pose_dir / 'read13_witness.json',
                witness(reread['molecular_system'], native13),
            )
            np.testing.assert_allclose(
                xyz(reread['molecular_system']),
                xyz(native13),
                rtol=0,
                atol=0.0000500001,
            )
            assert (
                reread['molecular_system'].molecular_mechanics.partial_charge is None
                and reread['molecular_system'].molecular_mechanics.atom_ff_type is None
            )
            for model, before in [
                (heavy, heavy_before),
                (full18, full_before),
                (typed18, typed_before),
                (native13, native_before),
            ]:
                assert model_snapshot(model) == before
            physical_values.append(
                dict(
                    profile=profile,
                    position=index,
                    coordinates_nm=xyz(native13).tolist(),
                    charges_e=charges(native13).tolist(),
                    types=prepared.atom_types,
                )
            )
            entry = dict(
                original_position=index,
                historical_pose_id=row['pose_id'],
                historical_scores=row['original_scores'],
                new_pose_id=f'e1-new-local-H-{index}',
                new_partner_state_id=prepared.state_id,
                native_export_sha256=mapping['payload_sha256'],
                dockingmt_export_sha256=hashlib.sha256(
                    dmt_payload.encode()
                ).hexdigest(),
                source_heavy_coordinates=row['coordinates'],
                historical_H_identity_admitted=False,
                minimum_backend_box_margin=dict(value=margin, unit='angstrom'),
                original_requested_box_margin=dict(
                    value=original_margin, unit='angstrom'
                ),
            )
            save(pose_dir / 'association.json', entry)
            rows.append(entry)
        assert model_snapshot(base) == base_before
        assert puw.configure.report() == profile_policy
        observation = dict(
            profile=profile,
            prepared_poses=rows,
            completed_pose_count=len(rows),
            source_checked_snapshot_unchanged=True,
        )
        save(directory / 'observations.json', observation)
        observations.append(observation)
    assert puw.configure.report() == caller_policy

In [ ]:
assert len(observations) == 2 and counts['get_atom_correspondences'] == 30
assert (
    counts['add_missing_hydrogens']
    == counts['assign_partial_charges']
    == counts['assign_autodock_atom_types']
    == counts['prepare_ligand']
    == 10
)
assert (
    len({row['dockingmt_export_sha256'] for row in observations[0]['prepared_poses']})
    == 5
)
for index in range(1, 6):
    for name in ['native.pdbqt', 'dockingmt.pdbqt']:
        assert (output / 'normal' / f'pose_{index}' / name).read_bytes() == (
            output / 'pm_fs' / f'pose_{index}' / name
        ).read_bytes()
    first = next(
        row
        for row in physical_values
        if row['profile'] == 'normal' and row['position'] == index
    )
    second = next(
        row
        for row in physical_values
        if row['profile'] == 'pm_fs' and row['position'] == index
    )
    np.testing.assert_allclose(
        first['coordinates_nm'], second['coordinates_nm'], rtol=0, atol=1e-12
    )
    np.testing.assert_allclose(
        first['charges_e'], second['charges_e'], rtol=0, atol=1e-12
    )
    assert first['types'] == second['types']
for name, path in inputs.items():
    assert path.read_bytes() == input_bytes[name]
assert not any(name == 'vina' or name.startswith('vina.') for name in sys.modules)
expected = dict(
    status='prospective_not_executed',
    reference_policy='native_current_conformation',
    version='1.2.7',
    cpu=1,
    seed=7,
    no_refine=False,
    force_even_voxels=False,
    grid_spacing=dict(value=0.375, unit='angstrom'),
    backend_box=dict(center=center.tolist(), size=size.tolist(), unit='angstrom'),
    original_backend_box=legacy['provenance']['backend_box'],
    receptor_sha256=hashlib.sha256(receptor).hexdigest(),
    receptor_preparation_assessment='unassessed',
    expected_factory_weights={
        'vina': [-0.035579, -0.005156, 0.840245, -0.035069, -0.587439, 50.0, 0.05846],
        'vinardo': [-0.045, 0.8, -0.035, -0.6, 50.0, 0.05846],
    },
    submitted_ligands=observations[0]['prepared_poses'],
    required_future_observations=[
        'actual native/Python artifacts',
        'native version/info weights and grid',
        'captured submitted bytes',
        'all eight components and reference records',
        'audits and histories',
    ],
    numeric_ordering='not admitted by current public rank_by for distinct described partner contexts',
)
save(output / 'prospective_scoring_contract.json', expected)
loaded_native = []
for name, module in list(sys.modules.items()):
    filename = getattr(module, '__file__', None)
    if name.startswith('molsysmt.') and filename and filename.endswith('.so'):
        path = Path(filename).resolve()
        loaded_native.append(
            dict(
                module=name,
                file=path.name,
                sha256=hashlib.sha256(path.read_bytes()).hexdigest(),
                inside_selected_provider=path.is_relative_to(
                    Path(msm.__file__).resolve().parent
                ),
                build_source_correspondence='unassessed',
            )
        )
receipt = dict(
    schema='dockingmt.e1_complete_pose_receiving@1',
    owner='uibcdf/dockingmt#78',
    python=platform.python_version(),
    versions={
        name: metadata.version(name)
        for name in [
            'dockingmt',
            'molsysmt',
            'argdigest',
            'pyunitwizard',
            'numpy',
            'rdkit',
            'h5py',
        ]
    },
    loaded_provider_native_modules=loaded_native,
    input_sha256=registered_inputs,
    reused_helper_cell_sha256=hashlib.sha256(helper_source.encode()).hexdigest(),
    profiles=observations,
    completed_profiles=2,
    new_declared_states=1,
    distinct_historical_heavy_coordinate_sets=5,
    completed_new_pose_preparations=10,
    positive_witnesses=30,
    monitored_direct_consumer_calls=counts,
    monitored_direct_consumer_budgets=budgets,
    receptor_input_sha256=hashlib.sha256(receptor).hexdigest(),
    new_energy_values=0,
    vina_calls=0,
    historical_H_identity_admitted=False,
    complete_MM_persistence_qualified=False,
    score_comparison_executed=False,
    e1_accepted=False,
    claim_limit='New full-pose construction with local H; prospective comparison contract only. No historical prepared identity, energies, numerical ordering or public artifact qualification.',
)
save(output / 'receipt.json', receipt)
print(
    json.dumps(
        dict(
            prepared_poses=10,
            distinct_coordinate_sets=5,
            positive_witnesses=30,
            vina_calls=0,
        )
    )
)